# Python orientado a objetos e SQL com SQLite

**Objetivo:** persistir objetos `Aluno`, `Disciplina` e `Matricula` em um banco relacional usando SQL escrito diretamente em Python.

Usaremos `sqlite3` (biblioteca padrão), SQL parametrizado, uma conexão compartilhada (Singleton) e CRUD completo. **Sem Pandas e sem SQLAlchemy.**

**No Google Colab:** `sqlite3` já está disponível: não é necessário instalar SQLite. `sqlite3.connect("universidade.db")` **cria o arquivo se ele não existir**. O arquivo fica no armazenamento temporário do Colab.

## 1. Conexão SQLite (Singleton)

`BancoSQLite()` sempre devolve a mesma instância enquanto a conexão está aberta. Assim, todos os objetos de acesso a dados compartilham uma única conexão.

In [ ]:
import sqlite3
from pathlib import Path

ARQUIVO_BANCO = Path("universidade.db")
existia_antes = ARQUIVO_BANCO.exists()


class BancoSQLite:
    _instancia = None

    def __new__(cls, arquivo=ARQUIVO_BANCO):
        if cls._instancia is None:
            banco = super().__new__(cls)
            banco.arquivo = Path(arquivo)
            banco.conexao = sqlite3.connect(banco.arquivo)
            banco.conexao.row_factory = sqlite3.Row
            banco.conexao.execute("PRAGMA foreign_keys = ON")
            cls._instancia = banco
        return cls._instancia

    def fechar(self):
        self.conexao.close()
        type(self)._instancia = None


banco = BancoSQLite()
conexao = banco.conexao

sql = "SELECT sqlite_version() AS versao"
resultado = conexao.execute(sql).fetchone()

print("Arquivo já existia?", existia_antes)
print("Arquivo existe agora?", ARQUIVO_BANCO.exists())
print("SQLite:", resultado["versao"])
print("Mesma conexão?", conexao is BancoSQLite().conexao)

assert ARQUIVO_BANCO.exists()
assert banco is BancoSQLite()
assert conexao is BancoSQLite().conexao

Arquivo já existia? False
Arquivo existe agora? True
SQLite: 3.45.1
Consulta de teste: 4
Mesma conexão? True


## 2. Criando as tabelas

- `alunos`: chave primária `nusp`;
- `disciplinas`: chave primária `codigo_disciplina`;
- `matriculas`: chave primária `id_matricula` e chaves estrangeiras.

A coluna `aprovado` não é armazenada: é calculada a partir da nota e da frequência.

**Para repetir esta aula:** a próxima célula remove e recria as tabelas. Em uma aplicação real, não apague dados dessa forma.

In [ ]:
# Reinicializar os dados ao executar o notebook desde o início.
#conexao.execute("DROP TABLE IF EXISTS matriculas")
#conexao.execute("DROP TABLE IF EXISTS disciplinas")
#conexao.execute("DROP TABLE IF EXISTS alunos")

sql_alunos = """
CREATE TABLE IF NOT EXISTS alunos (
    nusp INTEGER PRIMARY KEY,
    nome TEXT NOT NULL,
    email TEXT NOT NULL,
    curso TEXT NOT NULL,
    ano_ingresso INTEGER NOT NULL
)
"""

sql_disciplinas = """
CREATE TABLE IF NOT EXISTS disciplinas (
    codigo_disciplina TEXT PRIMARY KEY,
    nome_disciplina TEXT NOT NULL
)
"""

sql_matriculas = """
CREATE TABLE IF NOT EXISTS matriculas (
    id_matricula INTEGER PRIMARY KEY,
    nusp INTEGER NOT NULL,
    codigo_disciplina TEXT NOT NULL,
    periodo TEXT NOT NULL,
    nota REAL NOT NULL,
    frequencia REAL NOT NULL,
    FOREIGN KEY (nusp) REFERENCES alunos(nusp),
    FOREIGN KEY (codigo_disciplina) REFERENCES disciplinas(codigo_disciplina)
)
"""

conexao.execute(sql_alunos)
conexao.execute(sql_disciplinas)
conexao.execute(sql_matriculas)
conexao.commit()
print("Tabelas criadas.")

Tabelas criadas.


**Consulta de diagnóstico:** o próprio SQLite mantém informações sobre as tabelas no catálogo `sqlite_master`.

In [ ]:
sql = "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
for linha in conexao.execute(sql).fetchall():
    print(linha["name"])

alunos
disciplinas
matriculas


## 3. Classes do domínio

As classes representam os dados e o comportamento dos objetos. A persistência será feita com SQL explícito, em classes separadas.

In [ ]:
from abc import ABC, abstractmethod
from typing import Final


class Pessoa(ABC):
    def __init__(self, nusp: int, nome: str, email: str):
        self._nusp: Final[int] = nusp
        self.nome = nome
        self.email = email

    @property
    def nusp(self) -> int:
        return self._nusp

    @abstractmethod
    def to_dict(self) -> dict:
        pass

    def __repr__(self):
        return f"{type(self).__name__}(nusp={self.nusp}, nome={self.nome!r})"


class Aluno(Pessoa):
    def __init__(self, nusp: int, nome: str, email: str,
                 curso: str, ano_ingresso: int):
        super().__init__(nusp, nome, email)
        self.curso = curso
        self._ano_ingresso = ano_ingresso

    @property
    def ano_ingresso(self) -> int:
        return self._ano_ingresso

    def to_dict(self) -> dict:
        return {
            "nusp": self.nusp,
            "nome": self.nome,
            "email": self.email,
            "curso": self.curso,
            "ano_ingresso": self.ano_ingresso,
        }

In [ ]:
class Disciplina:
    def __init__(self, codigo: str, nome: str):
        self._codigo: Final[str] = codigo
        self.nome = nome

    @property
    def codigo(self) -> str:
        return self._codigo

    def to_dict(self) -> dict:
        return {
            "codigo_disciplina": self.codigo,
            "nome_disciplina": self.nome,
        }

    def __repr__(self):
        return f"Disciplina(codigo={self.codigo!r}, nome={self.nome!r})"

In [ ]:
class Matricula:
    def __init__(self, id_matricula: int, aluno: Aluno,
                 disciplina: Disciplina, periodo: str,
                 nota: float, frequencia: float):
        self._id_matricula: Final[int] = id_matricula
        self.aluno = aluno
        self.disciplina = disciplina
        self.periodo = periodo
        self.nota = nota
        self.frequencia = frequencia

    @property
    def id_matricula(self) -> int:
        return self._id_matricula

    @property
    def aprovado(self) -> bool:
        return self.nota >= 5.0 and self.frequencia >= 70.0

    def to_dict(self) -> dict:
        return {
            "id_matricula": self.id_matricula,
            "nusp": self.aluno.nusp,
            "codigo_disciplina": self.disciplina.codigo,
            "periodo": self.periodo,
            "nota": self.nota,
            "frequencia": self.frequencia,
            "aprovado": self.aprovado,
        }

    def resumo(self) -> str:
        situacao = "Aprovado" if self.aprovado else "Reprovado"
        return f"{self.aluno.nome} - {self.disciplina.nome}: {situacao}"

    def __repr__(self):
        return (
            f"Matricula(id={self.id_matricula}, "
            f"aluno={self.aluno.nusp}, "
            f"disciplina={self.disciplina.codigo})"
        )

### Teste dos objetos (ainda sem banco)

Os objetos podem ser criados e utilizados na memória, como na aula anterior.

In [ ]:
aluno_exemplo = Aluno(100, "Ana Souza", "ana@universidade.example", "Engenharia de Computação", 2026)
disciplina_exemplo = Disciplina("COMP301", "Inteligência Artificial")
matricula_exemplo = Matricula(1, aluno_exemplo, disciplina_exemplo, "2026-2", 8.0, 90.0)

print(aluno_exemplo)
print(aluno_exemplo.to_dict())
print(disciplina_exemplo)
print(matricula_exemplo)
print(matricula_exemplo.resumo())

assert matricula_exemplo.aluno is aluno_exemplo
assert matricula_exemplo.aprovado

Aluno(nusp=100, nome='Ana Souza')
{'nusp': 100, 'nome': 'Ana Souza', 'email': 'ana@universidade.example', 'curso': 'Engenharia de Computação', 'ano_ingresso': 2026}
Disciplina(codigo='COMP301', nome='Inteligência Artificial')
Matricula(id=1, aluno=100, disciplina=COMP301)
Ana Souza - Inteligência Artificial: Aprovado


## 4. Gerando dados fictícios

Usamos o Faker somente para gerar nomes. O banco continua sendo acessado exclusivamente com `sqlite3`.

In [ ]:
%pip -q install Faker

In [ ]:
import random
from faker import Faker

SEED = 42
N_ALUNOS = 60

CURSOS = [
    "Engenharia de Computação",
    "Engenharia Elétrica",
    "Ciência da Computação",
    "Sistemas de Informação",
    "Engenharia de Sistemas Digitais",
]

CATALOGO = [
    ("COMP101", "Programação em Python"),
    ("COMP102", "Algoritmos e Estruturas de Dados"),
    ("COMP201", "Banco de Dados"),
    ("COMP301", "Inteligência Artificial"),
    ("MAT101", "Matemática Discreta"),
    ("ENG201", "Engenharia de Software"),
]

ANOS = [2023, 2024, 2025, 2026]


def gerar_dados(n_alunos=60):
    fake = Faker("pt_BR")
    fake.seed_instance(SEED)
    rng = random.Random(SEED)

    disciplinas = [Disciplina(codigo, nome) for codigo, nome in CATALOGO]

    alunos = []
    for i in range(n_alunos):
        nusp = 95633 + i
        nome = fake.name()
        partes = nome.lower().split()
        email = f"{partes[0][0]}{partes[-1]}@usp.br"
        alunos.append(Aluno(
            nusp=nusp,
            nome=nome,
            email=email,
            curso=rng.choice(CURSOS),
            ano_ingresso=rng.choice(ANOS),
        ))

    matriculas = []
    proximo_id = 1
    for aluno in alunos:
        disciplinas_sorteadas = rng.sample(disciplinas, k=rng.randint(2, 4))
        for disciplina in disciplinas_sorteadas:
            matriculas.append(Matricula(
                id_matricula=proximo_id,
                aluno=aluno,
                disciplina=disciplina,
                periodo="2026-2",
                nota=round(rng.uniform(2.0, 10.0), 1),
                frequencia=round(rng.uniform(55.0, 100.0), 1),
            ))
            proximo_id += 1

    return alunos, disciplinas, matriculas

In [ ]:
alunos, disciplinas, matriculas = gerar_dados(N_ALUNOS)

print("Alunos em memória:", len(alunos))
print("Disciplinas em memória:", len(disciplinas))
print("Matrículas em memória:", len(matriculas))
print("Três alunos:", alunos[:3])
print("Três matrículas:", matriculas[:3])

Alunos em memória: 60
Disciplinas em memória: 6
Matrículas em memória: 179
Três alunos: [Aluno(nusp=95633, nome='Brenda Alves'), Aluno(nusp=95634, nome='Sra. Isabelly Câmara'), Aluno(nusp=95635, nome='Cauã Rocha')]
Três matrículas: [Matricula(id=1, aluno=95633, disciplina=COMP101), Matricula(id=2, aluno=95633, disciplina=ENG201), Matricula(id=3, aluno=95633, disciplina=COMP201)]


## 5. Consultas em listas Python

Primeiro, filtramos e ordenamos objetos na memória; depois veremos as consultas SQL equivalentes.

In [ ]:
# Filtro em Python (filter ou compreensão de lista)
alunos_comp = [a for a in alunos if a.curso == "Engenharia de Computação"]

# Projeção dos nomes (map)
nomes_comp = list(map(lambda a: a.nome, alunos_comp))

print("Alunos de Computação:", len(alunos_comp))
print("Primeiros nomes:", nomes_comp[:5])

Alunos de Computação: 13
Primeiros nomes: ['Brenda Alves', 'Dra. Aurora Pastor', 'Henrique da Conceição', 'Amanda Novais', 'Sr. Caleb Garcia']


In [ ]:
# Ordenação em Python (sorted)
matriculas_ordenadas = sorted(
    matriculas,
    key=lambda m: (-m.nota, m.aluno.nome, m.id_matricula),
)
for m in matriculas_ordenadas[:5]:
    print(m.resumo())

Eduardo Cunha - Inteligência Artificial: Reprovado
Luiz Miguel Oliveira - Programação em Python: Aprovado
Brenda Alves - Inteligência Artificial: Aprovado
Maria Clara Castro - Algoritmos e Estruturas de Dados: Reprovado
Sr. Caleb Garcia - Matemática Discreta: Aprovado


## 6. Persistência manual: DAO

`AlunoDAO`, `DisciplinaDAO` e `MatriculaDAO` executam `INSERT`, `SELECT`, `UPDATE` e `DELETE` escritos como strings. Cada DAO usa a **mesma conexão** fornecida pelo Singleton.

In [ ]:
class AlunoDAO:
    def __init__(self):
        self.conexao = BancoSQLite().conexao

    def inserir(self, aluno: Aluno):
        sql = """
        INSERT INTO alunos (nusp, nome, email, curso, ano_ingresso)
        VALUES (?, ?, ?, ?, ?)
        """
        valores = (aluno.nusp, aluno.nome, aluno.email,
                   aluno.curso, aluno.ano_ingresso)
        self.conexao.execute(sql, valores)
        self.conexao.commit()

    def buscar(self, nusp: int):
        sql = "SELECT * FROM alunos WHERE nusp = ?"
        linha = self.conexao.execute(sql, (nusp,)).fetchone()
        if linha is None:
            return None
        return Aluno(linha["nusp"], linha["nome"], linha["email"],
                     linha["curso"], linha["ano_ingresso"])

    def listar_por_curso(self, curso: str):
        sql = "SELECT * FROM alunos WHERE curso = ? ORDER BY nome"
        linhas = self.conexao.execute(sql, (curso,)).fetchall()
        return [Aluno(linha["nusp"], linha["nome"], linha["email"],
                      linha["curso"], linha["ano_ingresso"])
                for linha in linhas]

    def atualizar(self, aluno: Aluno):
        sql = """
        UPDATE alunos
        SET nome = ?, email = ?, curso = ?, ano_ingresso = ?
        WHERE nusp = ?
        """
        valores = (aluno.nome, aluno.email, aluno.curso,
                   aluno.ano_ingresso, aluno.nusp)
        cursor = self.conexao.execute(sql, valores)
        self.conexao.commit()
        return cursor.rowcount

    def excluir(self, nusp: int):
        sql = "DELETE FROM alunos WHERE nusp = ?"
        cursor = self.conexao.execute(sql, (nusp,))
        self.conexao.commit()
        return cursor.rowcount

`?` é um **parâmetro SQL**: os valores são enviados separadamente, por exemplo `(nusp,)`. A instrução é preparada internamente pelo SQLite. Assim, dados de entrada não são interpretados como trechos do SQL.

In [ ]:
class DisciplinaDAO:
    def __init__(self):
        self.conexao = BancoSQLite().conexao

    def inserir(self, disciplina: Disciplina):
        sql = """
        INSERT INTO disciplinas (codigo_disciplina, nome_disciplina)
        VALUES (?, ?)
        """
        self.conexao.execute(sql, (disciplina.codigo, disciplina.nome))
        self.conexao.commit()

    def buscar(self, codigo: str):
        sql = "SELECT * FROM disciplinas WHERE codigo_disciplina = ?"
        linha = self.conexao.execute(sql, (codigo,)).fetchone()
        if linha is None:
            return None
        return Disciplina(linha["codigo_disciplina"], linha["nome_disciplina"])

    def atualizar(self, disciplina: Disciplina):
        sql = """
        UPDATE disciplinas SET nome_disciplina = ?
        WHERE codigo_disciplina = ?
        """
        cursor = self.conexao.execute(sql, (disciplina.nome, disciplina.codigo))
        self.conexao.commit()
        return cursor.rowcount

    def excluir(self, codigo: str):
        sql = "DELETE FROM disciplinas WHERE codigo_disciplina = ?"
        cursor = self.conexao.execute(sql, (codigo,))
        self.conexao.commit()
        return cursor.rowcount


In [ ]:
class MatriculaDAO:
    def __init__(self):
        self.conexao = BancoSQLite().conexao

    def inserir(self, matricula: Matricula):
        sql = """
        INSERT INTO matriculas
            (id_matricula, nusp, codigo_disciplina, periodo, nota, frequencia)
        VALUES (?, ?, ?, ?, ?, ?)
        """
        valores = (matricula.id_matricula, matricula.aluno.nusp,
                   matricula.disciplina.codigo, matricula.periodo,
                   matricula.nota, matricula.frequencia)
        self.conexao.execute(sql, valores)
        self.conexao.commit()

    def buscar(self, id_matricula: int):
        sql = "SELECT * FROM matriculas WHERE id_matricula = ?"
        linha = self.conexao.execute(sql, (id_matricula,)).fetchone()
        if linha is None:
            return None

        aluno = AlunoDAO().buscar(linha["nusp"])
        disciplina = DisciplinaDAO().buscar(linha["codigo_disciplina"])
        return Matricula(linha["id_matricula"], aluno, disciplina,
                         linha["periodo"], linha["nota"], linha["frequencia"])

    def atualizar(self, matricula: Matricula):
        sql = """
        UPDATE matriculas SET periodo = ?, nota = ?, frequencia = ?
        WHERE id_matricula = ?
        """
        valores = (matricula.periodo, matricula.nota,
                   matricula.frequencia, matricula.id_matricula)
        cursor = self.conexao.execute(sql, valores)
        self.conexao.commit()
        return cursor.rowcount

    def excluir(self, id_matricula: int):
        sql = "DELETE FROM matriculas WHERE id_matricula = ?"
        cursor = self.conexao.execute(sql, (id_matricula,))
        self.conexao.commit()
        return cursor.rowcount

### Gravando os objetos

Gravamos disciplinas e alunos antes das matrículas para respeitar as chaves estrangeiras. Cada operação de escrita é confirmada com `commit()`.

In [ ]:
aluno_dao = AlunoDAO()
disciplina_dao = DisciplinaDAO()
matricula_dao = MatriculaDAO()

assert aluno_dao.conexao is disciplina_dao.conexao is matricula_dao.conexao

for disciplina in disciplinas:
    disciplina_dao.inserir(disciplina)

for aluno in alunos:
    aluno_dao.inserir(aluno)

for matricula in matriculas:
    matricula_dao.inserir(matricula)

print("Dados gravados com uma conexão compartilhada.")

Dados gravados com uma conexão compartilhada.


In [ ]:
# Os nomes das tabelas vêm desta lista fixa, não de entrada externa.
for tabela in ("alunos", "disciplinas", "matriculas"):
    quantidade = conexao.execute(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    print(f"{tabela}: {quantidade}")

assert conexao.execute("SELECT COUNT(*) FROM alunos").fetchone()[0] == len(alunos)
assert conexao.execute("SELECT COUNT(*) FROM matriculas").fetchone()[0] == len(matriculas)

alunos: 60
disciplinas: 6
matriculas: 179


## 7. Consultas SQL

Usaremos `SELECT`, `WHERE`, `ORDER BY`, `JOIN` e `GROUP BY`. Primeiro, uma consulta com parâmetro posicional (`?`).

In [ ]:
# Equivalente ao filtro Python por curso.
sql = """
SELECT nusp, nome, curso
FROM alunos
WHERE curso = ?
ORDER BY nome
"""

curso = "Engenharia de Computação"
cursor = conexao.cursor()
cursor.execute(sql, (curso,))
linhas = cursor.fetchall()

print("Quantidade:", len(linhas))
for linha in linhas[:5]:
    print(dict(linha))

assert len(linhas) == len(alunos_comp)

Quantidade: 13
{'nusp': 95639, 'nome': 'Amanda Novais', 'curso': 'Engenharia de Computação'}
{'nusp': 95649, 'nome': 'Anna Liz Novaes', 'curso': 'Engenharia de Computação'}
{'nusp': 95633, 'nome': 'Brenda Alves', 'curso': 'Engenharia de Computação'}
{'nusp': 95653, 'nome': 'Bruno da Conceição', 'curso': 'Engenharia de Computação'}
{'nusp': 95667, 'nome': 'Davi Miguel da Cruz', 'curso': 'Engenharia de Computação'}


### Parâmetros nomeados (`:curso`)

Também podemos usar um dicionário para associar nomes aos parâmetros.

In [ ]:
sql = "SELECT nusp, nome FROM alunos WHERE curso = :curso ORDER BY nome"
linhas = conexao.execute(sql, {"curso": "Engenharia Elétrica"}).fetchall()
for linha in linhas[:3]:
    print(dict(linha))

{'nusp': 95688, 'nome': 'Agatha Peixoto'}
{'nusp': 95635, 'nome': 'Cauã Rocha'}
{'nusp': 95669, 'nome': 'Dra. Mariane Pacheco'}


### Alunos aprovados

A aprovação é calculada pela condição `nota >= 5` e `frequencia >= 70`.

In [ ]:
sql = """
SELECT id_matricula, nusp, codigo_disciplina, nota, frequencia
FROM matriculas
WHERE nota >= 5.0 AND frequencia >= 70.0
ORDER BY nota DESC, id_matricula ASC
"""

aprovados_sql = conexao.execute(sql).fetchall()
aprovados_python = [m for m in matriculas if m.aprovado]
print("Aprovados pelo SQL:", len(aprovados_sql))
print("Aprovados pelo Python:", len(aprovados_python))
assert len(aprovados_sql) == len(aprovados_python)
for linha in aprovados_sql[:5]:
    print(dict(linha))

Aprovados pelo SQL: 72
Aprovados pelo Python: 72
{'id_matricula': 55, 'nusp': 95650, 'codigo_disciplina': 'COMP101', 'nota': 9.9, 'frequencia': 84.4}
{'id_matricula': 4, 'nusp': 95633, 'codigo_disciplina': 'COMP301', 'nota': 9.8, 'frequencia': 89.3}
{'id_matricula': 38, 'nusp': 95645, 'codigo_disciplina': 'MAT101', 'nota': 9.7, 'frequencia': 81.1}
{'id_matricula': 6, 'nusp': 95634, 'codigo_disciplina': 'COMP201', 'nota': 9.6, 'frequencia': 96.5}
{'id_matricula': 65, 'nusp': 95654, 'codigo_disciplina': 'COMP301', 'nota': 9.6, 'frequencia': 86.1}


### `JOIN`: aprovados em Inteligência Artificial

As chaves estrangeiras permitem relacionar matrículas, alunos e disciplinas.

In [ ]:
sql = """
SELECT a.nusp, a.nome, a.curso,
       d.nome_disciplina, m.nota, m.frequencia
FROM matriculas AS m
JOIN alunos AS a ON m.nusp = a.nusp
JOIN disciplinas AS d ON m.codigo_disciplina = d.codigo_disciplina
WHERE d.codigo_disciplina = ?
  AND m.nota >= 5.0
  AND m.frequencia >= 70.0
ORDER BY m.nota DESC, a.nome ASC, m.id_matricula ASC
"""

consulta_ia_sql = conexao.execute(sql, ("COMP301",)).fetchall()
consulta_ia_python = [m for m in matriculas
                      if m.disciplina.codigo == "COMP301" and m.aprovado]

print("Aprovados em IA pelo SQL:", len(consulta_ia_sql))
print("Aprovados em IA pelo Python:", len(consulta_ia_python))
assert len(consulta_ia_sql) == len(consulta_ia_python)

for linha in consulta_ia_sql[:5]:
    print(dict(linha))

Aprovados em IA pelo SQL: 13
Aprovados em IA pelo Python: 13
{'nusp': 95633, 'nome': 'Brenda Alves', 'curso': 'Engenharia de Computação', 'nome_disciplina': 'Inteligência Artificial', 'nota': 9.8, 'frequencia': 89.3}
{'nusp': 95654, 'nome': 'Mateus da Mota', 'curso': 'Engenharia de Sistemas Digitais', 'nome_disciplina': 'Inteligência Artificial', 'nota': 9.6, 'frequencia': 86.1}
{'nusp': 95636, 'nome': 'Dra. Aurora Pastor', 'curso': 'Engenharia de Computação', 'nome_disciplina': 'Inteligência Artificial', 'nota': 9.4, 'frequencia': 89.0}
{'nusp': 95673, 'nome': 'Emanuel Vieira', 'curso': 'Engenharia Elétrica', 'nome_disciplina': 'Inteligência Artificial', 'nota': 8.7, 'frequencia': 72.5}
{'nusp': 95674, 'nome': 'Mariana Peixoto', 'curso': 'Ciência da Computação', 'nome_disciplina': 'Inteligência Artificial', 'nota': 8.4, 'frequencia': 95.4}


### `GROUP BY`: alunos por curso

In [ ]:
sql = """
SELECT curso, COUNT(*) AS quantidade
FROM alunos
GROUP BY curso
ORDER BY quantidade DESC, curso ASC
"""

for linha in conexao.execute(sql).fetchall():
    print(f"{linha['curso']}: {linha['quantidade']}")

Engenharia de Sistemas Digitais: 17
Engenharia Elétrica: 14
Engenharia de Computação: 13
Ciência da Computação: 10
Sistemas de Informação: 6


## 8. SQL injection e consultas parametrizadas

Concatenar dados de um usuário no SQL pode alterar a consulta. No exemplo, usamos somente `SELECT` para comparar uma consulta **insegura** com outra **parametrizada**.

In [ ]:
entrada_usuario = "' OR 1=1 --"

# Inseguro: a entrada modifica o significado do SQL.
sql_inseguro = f"SELECT nusp FROM alunos WHERE nome = '{entrada_usuario}'"
resultado_inseguro = conexao.execute(sql_inseguro).fetchall()

# Seguro: a entrada é tratada como um valor literal.
sql_seguro = "SELECT nusp FROM alunos WHERE nome = ?"
resultado_seguro = conexao.execute(sql_seguro, (entrada_usuario,)).fetchall()

print("SQL inseguro:", sql_inseguro)
print("Resultados (inseguro):", len(resultado_inseguro))
print("Resultados (seguro):", len(resultado_seguro))
assert len(resultado_inseguro) == len(alunos)
assert len(resultado_seguro) == 0

SQL inseguro: SELECT nusp FROM alunos WHERE nome = '' OR 1=1 --'
Resultados (inseguro): 60
Resultados (seguro): 0


## 9. CRUD completo

Vamos inserir, consultar, atualizar e excluir objetos de cada entidade, preservando os dados iniciais.

In [ ]:
# C — CREATE
aluno_novo = Aluno(88888888, "Marina Oliveira", "marina@usp.br",
                   "Engenharia de Computação", 2026)
aluno_dao.inserir(aluno_novo)

# R — READ (o método cria um novo objeto a partir da linha do banco)
aluno_lido = aluno_dao.buscar(88888888)
print("Lido do banco:", aluno_lido.to_dict())

# U — UPDATE
aluno_lido.email = "marina.oliveira@usp.br"
linhas_alteradas = aluno_dao.atualizar(aluno_lido)
print("Linhas atualizadas:", linhas_alteradas)
print("Email após UPDATE:", aluno_dao.buscar(88888888).email)

# D — DELETE
linhas_excluidas = aluno_dao.excluir(88888888)
print("Linhas excluídas:", linhas_excluidas)
print("Busca após DELETE:", aluno_dao.buscar(88888888))

assert aluno_dao.buscar(88888888) is None

Lido do banco: {'nusp': 88888888, 'nome': 'Marina Oliveira', 'email': 'marina@usp.br', 'curso': 'Engenharia de Computação', 'ano_ingresso': 2026}
Linhas atualizadas: 1
Email após UPDATE: marina.oliveira@usp.br
Linhas excluídas: 1
Busca após DELETE: None


### CRUD de Disciplina

In [ ]:
# C e R — CREATE e READ
nova_disciplina = Disciplina("OPT999", "Laboratório de SQL")
disciplina_dao.inserir(nova_disciplina)
print("Após INSERT:", disciplina_dao.buscar("OPT999"))

# U — UPDATE
nova_disciplina.nome = "Laboratório de Banco de Dados"
print("Linhas atualizadas:", disciplina_dao.atualizar(nova_disciplina))
print("Após UPDATE:", disciplina_dao.buscar("OPT999").to_dict())

Após INSERT: Disciplina(codigo='OPT999', nome='Laboratório de SQL')
Linhas atualizadas: 1
Após UPDATE: {'codigo_disciplina': 'OPT999', 'nome_disciplina': 'Laboratório de Banco de Dados'}


### CRUD de Matricula

No Python, a matrícula guarda objetos; no SQLite, armazena as chaves estrangeiras.

In [ ]:
# C — CREATE
matricula_teste = Matricula(
    id_matricula=99999,
    aluno=alunos[0],
    disciplina=nova_disciplina,
    periodo="2026-2",
    nota=7.0,
    frequencia=90.0,
)
matricula_dao.inserir(matricula_teste)

# R — READ
matricula_lida = matricula_dao.buscar(99999)
print("Matrícula lida:", matricula_lida.to_dict())
print("Resumo:", matricula_lida.resumo())

# U — UPDATE
matricula_lida.nota = 4.0
print("Linhas atualizadas:", matricula_dao.atualizar(matricula_lida))
print("Aprovado após UPDATE?", matricula_dao.buscar(99999).aprovado)

# D — DELETE
print("Linhas excluídas:", matricula_dao.excluir(99999))
assert matricula_dao.buscar(99999) is None

Matrícula lida: {'id_matricula': 99999, 'nusp': 95633, 'codigo_disciplina': 'OPT999', 'periodo': '2026-2', 'nota': 7.0, 'frequencia': 90.0, 'aprovado': True}
Resumo: Brenda Alves - Laboratório de Banco de Dados: Aprovado
Linhas atualizadas: 1
Aprovado após UPDATE? False
Linhas excluídas: 1


In [ ]:
print("Disciplinas excluídas:", disciplina_dao.excluir("OPT999"))
assert disciplina_dao.buscar("OPT999") is None

Disciplinas excluídas: 1


### Integridade referencial

Uma disciplina com matrículas não pode ser excluída enquanto as chaves estrangeiras estiverem ativas.

In [ ]:
try:
    disciplina_dao.excluir("COMP301")
except sqlite3.IntegrityError as erro:
    conexao.rollback()
    print("Exclusão bloqueada:", erro)

assert disciplina_dao.buscar("COMP301") is not None

Exclusão bloqueada: FOREIGN KEY constraint failed


## 10. Persistência: fechar e reabrir

Fechamos o Singleton e criamos outro, que lê o mesmo arquivo `universidade.db`. Os registros continuam disponíveis.

In [ ]:
conexao_anterior = conexao
banco.fechar()

banco = BancoSQLite()
conexao = banco.conexao
assert conexao is not conexao_anterior
assert conexao is BancoSQLite().conexao

sql = "SELECT COUNT(*) AS quantidade FROM alunos"
linha = conexao.execute(sql).fetchone()
print("Alunos recuperados após reabrir:", linha["quantidade"])
assert linha["quantidade"] == N_ALUNOS

# Os DAOs novos usam a conexão reaberta.
aluno_dao = AlunoDAO()
disciplina_dao = DisciplinaDAO()
matricula_dao = MatriculaDAO()

print("Aluno recuperado:", aluno_dao.buscar(alunos[0].nusp))

Alunos recuperados após reabrir: 60
Aluno recuperado: Aluno(nusp=95633, nome='Brenda Alves')


## 11. Consultas e operações adicionais resolvidas

Exemplos complementares com `SELECT`, consulta parametrizada, `JOIN`, agregação e um ciclo completo de inserção/alteração/remoção envolvendo aluno e matrícula.

### `SELECT`: ingressantes em 2025

In [ ]:
sql = """
SELECT nusp, nome, ano_ingresso
FROM alunos
WHERE ano_ingresso = ?
ORDER BY nome
"""

for linha in conexao.execute(sql, (2025,)).fetchall()[:10]:
    print(dict(linha))

{'nusp': 95651, 'nome': 'Bernardo Aparecida', 'ano_ingresso': 2025}
{'nusp': 95657, 'nome': 'Brayan Pereira', 'ano_ingresso': 2025}
{'nusp': 95656, 'nome': 'Dr. Gustavo Henrique Rios', 'ano_ingresso': 2025}
{'nusp': 95644, 'nome': 'Eduardo Cunha', 'ano_ingresso': 2025}
{'nusp': 95652, 'nome': 'Heloísa Vargas', 'ano_ingresso': 2025}
{'nusp': 95679, 'nome': 'Juliana Vargas', 'ano_ingresso': 2025}
{'nusp': 95660, 'nome': 'Júlia Porto', 'ano_ingresso': 2025}
{'nusp': 95646, 'nome': 'Luiz Henrique Ferreira', 'ano_ingresso': 2025}
{'nusp': 95670, 'nome': 'Maria Cecília Câmara', 'ano_ingresso': 2025}
{'nusp': 95666, 'nome': 'Natália Casa Grande', 'ano_ingresso': 2025}


### `WHERE`: matrículas de um aluno

In [ ]:
nusp_consultado = alunos[0].nusp
sql = """
SELECT id_matricula, nusp, codigo_disciplina, nota
FROM matriculas
WHERE nusp = ?
ORDER BY codigo_disciplina
"""

for linha in conexao.execute(sql, (nusp_consultado,)).fetchall():
    print(dict(linha))

{'id_matricula': 1, 'nusp': 95633, 'codigo_disciplina': 'COMP101', 'nota': 4.7}
{'id_matricula': 3, 'nusp': 95633, 'codigo_disciplina': 'COMP201', 'nota': 7.8}
{'id_matricula': 4, 'nusp': 95633, 'codigo_disciplina': 'COMP301', 'nota': 9.8}
{'id_matricula': 2, 'nusp': 95633, 'codigo_disciplina': 'ENG201', 'nota': 3.3}


### `JOIN`: alunos matriculados em Banco de Dados

In [ ]:
sql = """
SELECT a.nome AS aluno, d.nome_disciplina AS disciplina, m.nota
FROM matriculas AS m
JOIN alunos AS a ON a.nusp = m.nusp
JOIN disciplinas AS d ON d.codigo_disciplina = m.codigo_disciplina
WHERE d.codigo_disciplina = ?
ORDER BY m.nota DESC
"""

for linha in conexao.execute(sql, ("COMP201",)).fetchall()[:10]:
    print(dict(linha))

{'aluno': 'Sra. Isabelly Câmara', 'disciplina': 'Banco de Dados', 'nota': 9.6}
{'aluno': 'Matheus Garcia', 'disciplina': 'Banco de Dados', 'nota': 9.4}
{'aluno': 'Thomas Monteiro', 'disciplina': 'Banco de Dados', 'nota': 9.1}
{'aluno': 'Luiz Miguel Oliveira', 'disciplina': 'Banco de Dados', 'nota': 9.0}
{'aluno': 'Sra. Ana Lívia Teixeira', 'disciplina': 'Banco de Dados', 'nota': 9.0}
{'aluno': 'Yuri Albuquerque', 'disciplina': 'Banco de Dados', 'nota': 8.7}
{'aluno': 'Manuela Camargo', 'disciplina': 'Banco de Dados', 'nota': 8.2}
{'aluno': 'Brenda Alves', 'disciplina': 'Banco de Dados', 'nota': 7.8}
{'aluno': 'Dra. Aurora Pastor', 'disciplina': 'Banco de Dados', 'nota': 7.5}
{'aluno': 'Henrique da Conceição', 'disciplina': 'Banco de Dados', 'nota': 7.4}


### `GROUP BY`: média de notas por disciplina

In [ ]:
sql = """
SELECT d.nome_disciplina, ROUND(AVG(m.nota), 2) AS media_notas
FROM disciplinas AS d
JOIN matriculas AS m ON m.codigo_disciplina = d.codigo_disciplina
GROUP BY d.codigo_disciplina, d.nome_disciplina
ORDER BY media_notas DESC
"""

for linha in conexao.execute(sql).fetchall():
    print(dict(linha))

{'nome_disciplina': 'Engenharia de Software', 'media_notas': 6.71}
{'nome_disciplina': 'Inteligência Artificial', 'media_notas': 6.42}
{'nome_disciplina': 'Banco de Dados', 'media_notas': 6.14}
{'nome_disciplina': 'Programação em Python', 'media_notas': 5.89}
{'nome_disciplina': 'Algoritmos e Estruturas de Dados', 'media_notas': 5.86}
{'nome_disciplina': 'Matemática Discreta', 'media_notas': 5.78}


### CRUD integrado: aluno e matrícula

In [ ]:
# Criar aluno e matrícula.
aluno_teste = Aluno(77777777, "Pedro Lima", "pedro@usp.br", "Engenharia Elétrica", 2025)
aluno_dao.inserir(aluno_teste)
disciplina_existente = disciplina_dao.buscar("COMP201")

matricula_teste = Matricula(99998, aluno_teste, disciplina_existente, "2026-2", 6.0, 85.0)
matricula_dao.inserir(matricula_teste)
print("Inserido:", matricula_dao.buscar(99998).to_dict())

# Alterar a nota e verificar a gravação.
matricula_teste.nota = 9.0
matricula_dao.atualizar(matricula_teste)
print("Nota atualizada:", matricula_dao.buscar(99998).nota)
assert matricula_dao.buscar(99998).nota == 9.0

# Remover a matrícula antes do aluno.
matricula_dao.excluir(99998)
aluno_dao.excluir(77777777)
assert matricula_dao.buscar(99998) is None
assert aluno_dao.buscar(77777777) is None
print("Registros de teste removidos.")

Inserido: {'id_matricula': 99998, 'nusp': 77777777, 'codigo_disciplina': 'COMP201', 'periodo': '2026-2', 'nota': 6.0, 'frequencia': 85.0, 'aprovado': True}
Nota atualizada: 9.0
Registros de teste removidos.


In [ ]:

class AlunoDAO:
    def __init__(self):
        self.conexao = BancoSQLite().conexao

    @staticmethod
    def _da_linha(linha):
        return Aluno(
            linha["nusp"],
            linha["nome"],
            linha["email"],
            linha["curso"],
            linha["ano_ingresso"]
        )

    def inserir(self, aluno: Aluno):
        sql = """
        INSERT INTO alunos (nusp, nome, email, curso, ano_ingresso)
        VALUES (?, ?, ?, ?, ?)
        """
        valores = (
            aluno.nusp, aluno.nome, aluno.email,
            aluno.curso, aluno.ano_ingresso
        )
        self.conexao.execute(sql, valores)
        self.conexao.commit()

    def buscar(self, nusp: int):
        sql = "SELECT * FROM alunos WHERE nusp = ?"
        linha = self.conexao.execute(sql, (nusp,)).fetchone()

        if linha is None:
            return None
        return self._da_linha(linha)

    def listar(self):
        sql = "SELECT * FROM alunos ORDER BY nome"
        linhas = self.conexao.execute(sql).fetchall()
        return [self._da_linha(linha) for linha in linhas]

    def listar_por_curso(self, curso: str):
        sql = """
        SELECT *
        FROM alunos
        WHERE curso = ?
        ORDER BY nome
        """
        linhas = self.conexao.execute(sql, (curso,)).fetchall()
        return [self._da_linha(linha) for linha in linhas]

    def consultar_por_curso(self, curso: str):
        sql = """
        SELECT nusp, nome, curso
        FROM alunos
        WHERE curso = ?
        ORDER BY nome
        """
        linhas = self.conexao.execute(sql, (curso,)).fetchall()
        return [dict(linha) for linha in linhas]

    def consultar_por_curso_nomeado(self, curso: str):
        sql = """
        SELECT nusp, nome
        FROM alunos
        WHERE curso = :curso
        ORDER BY nome
        """
        linhas = self.conexao.execute(
            sql, {"curso": curso}
        ).fetchall()
        return [dict(linha) for linha in linhas]

    def listar_ingressantes(self, ano: int):
        sql = """
        SELECT nusp, nome, ano_ingresso
        FROM alunos
        WHERE ano_ingresso = ?
        ORDER BY nome
        """
        linhas = self.conexao.execute(sql, (ano,)).fetchall()
        return [dict(linha) for linha in linhas]

    def contar_por_curso(self):
        sql = """
        SELECT curso, COUNT(*) AS quantidade
        FROM alunos
        GROUP BY curso
        ORDER BY quantidade DESC, curso ASC
        """
        linhas = self.conexao.execute(sql).fetchall()
        return [dict(linha) for linha in linhas]

    def contar(self):
        sql = "SELECT COUNT(*) AS quantidade FROM alunos"
        linha = self.conexao.execute(sql).fetchone()
        return linha["quantidade"]

    def buscar_por_nome(self, nome: str):
        sql = "SELECT nusp FROM alunos WHERE nome = ?"
        linhas = self.conexao.execute(sql, (nome,)).fetchall()
        return [linha["nusp"] for linha in linhas]

    def atualizar(self, aluno: Aluno):
        sql = """
        UPDATE alunos
        SET nome = ?, email = ?, curso = ?, ano_ingresso = ?
        WHERE nusp = ?
        """
        valores = (
            aluno.nome, aluno.email, aluno.curso,
            aluno.ano_ingresso, aluno.nusp
        )
        cursor = self.conexao.execute(sql, valores)
        self.conexao.commit()
        return cursor.rowcount

    def excluir(self, nusp: int):
        sql = "DELETE FROM alunos WHERE nusp = ?"
        cursor = self.conexao.execute(sql, (nusp,))
        self.conexao.commit()
        return cursor.rowcount


In [ ]:

class DisciplinaDAO:
    def __init__(self):
        self.conexao = BancoSQLite().conexao

    @staticmethod
    def _da_linha(linha):
        return Disciplina(
            linha["codigo_disciplina"],
            linha["nome_disciplina"]
        )

    def inserir(self, disciplina: Disciplina):
        sql = """
        INSERT INTO disciplinas (codigo_disciplina, nome_disciplina)
        VALUES (?, ?)
        """
        self.conexao.execute(
            sql, (disciplina.codigo, disciplina.nome)
        )
        self.conexao.commit()

    def buscar(self, codigo: str):
        sql = """
        SELECT *
        FROM disciplinas
        WHERE codigo_disciplina = ?
        """
        linha = self.conexao.execute(sql, (codigo,)).fetchone()

        if linha is None:
            return None
        return self._da_linha(linha)

    def listar(self):
        sql = """
        SELECT *
        FROM disciplinas
        ORDER BY nome_disciplina
        """
        linhas = self.conexao.execute(sql).fetchall()
        return [self._da_linha(linha) for linha in linhas]

    def buscar_por_nusp(self, nusp: int):
        sql = """
        SELECT DISTINCT d.codigo_disciplina, d.nome_disciplina
        FROM disciplinas AS d
        JOIN matriculas AS m
            ON d.codigo_disciplina = m.codigo_disciplina
        WHERE m.nusp = ?
        ORDER BY d.nome_disciplina
        """
        linhas = self.conexao.execute(sql, (nusp,)).fetchall()
        return [self._da_linha(linha) for linha in linhas]

    def listar_alunos_matriculados(self, codigo: str):
        sql = """
        SELECT a.nome AS aluno,
               d.nome_disciplina AS disciplina,
               m.nota
        FROM matriculas AS m
        JOIN alunos AS a
            ON a.nusp = m.nusp
        JOIN disciplinas AS d
            ON d.codigo_disciplina = m.codigo_disciplina
        WHERE d.codigo_disciplina = ?
        ORDER BY m.nota DESC
        """
        linhas = self.conexao.execute(sql, (codigo,)).fetchall()
        return [dict(linha) for linha in linhas]

    def medias_notas(self):
        sql = """
        SELECT d.nome_disciplina,
               ROUND(AVG(m.nota), 2) AS media_notas
        FROM disciplinas AS d
        JOIN matriculas AS m
            ON m.codigo_disciplina = d.codigo_disciplina
        GROUP BY d.codigo_disciplina, d.nome_disciplina
        ORDER BY media_notas DESC
        """
        linhas = self.conexao.execute(sql).fetchall()
        return [dict(linha) for linha in linhas]

    def atualizar(self, disciplina: Disciplina):
        sql = """
        UPDATE disciplinas
        SET nome_disciplina = ?
        WHERE codigo_disciplina = ?
        """
        cursor = self.conexao.execute(
            sql, (disciplina.nome, disciplina.codigo)
        )
        self.conexao.commit()
        return cursor.rowcount

    def excluir(self, codigo: str):
        sql = """
        DELETE FROM disciplinas
        WHERE codigo_disciplina = ?
        """
        cursor = self.conexao.execute(sql, (codigo,))
        self.conexao.commit()
        return cursor.rowcount


In [ ]:

class MatriculaDAO:
    def __init__(self):
        self.conexao = BancoSQLite().conexao

    def _da_linha(self, linha):
        aluno = AlunoDAO().buscar(linha["nusp"])
        disciplina = DisciplinaDAO().buscar(
            linha["codigo_disciplina"]
        )

        return Matricula(
            linha["id_matricula"],
            aluno,
            disciplina,
            linha["periodo"],
            linha["nota"],
            linha["frequencia"]
        )

    def inserir(self, matricula: Matricula):
        sql = """
        INSERT INTO matriculas
            (id_matricula, nusp, codigo_disciplina,
             periodo, nota, frequencia)
        VALUES (?, ?, ?, ?, ?, ?)
        """
        valores = (
            matricula.id_matricula,
            matricula.aluno.nusp,
            matricula.disciplina.codigo,
            matricula.periodo,
            matricula.nota,
            matricula.frequencia
        )
        self.conexao.execute(sql, valores)
        self.conexao.commit()

    def buscar(self, id_matricula: int):
        sql = """
        SELECT *
        FROM matriculas
        WHERE id_matricula = ?
        """
        linha = self.conexao.execute(
            sql, (id_matricula,)
        ).fetchone()

        if linha is None:
            return None
        return self._da_linha(linha)

    def listar(self):
        sql = """
        SELECT *
        FROM matriculas
        ORDER BY id_matricula
        """
        linhas = self.conexao.execute(sql).fetchall()
        return [self._da_linha(linha) for linha in linhas]

    def listar_por_nusp(self, nusp: int):
        sql = """
        SELECT id_matricula, nusp, codigo_disciplina, nota
        FROM matriculas
        WHERE nusp = ?
        ORDER BY codigo_disciplina
        """
        linhas = self.conexao.execute(sql, (nusp,)).fetchall()
        return [dict(linha) for linha in linhas]

    def listar_aprovados(
        self, nota_minima=5.0, frequencia_minima=70.0
    ):
        sql = """
        SELECT id_matricula, nusp, codigo_disciplina,
               nota, frequencia
        FROM matriculas
        WHERE nota >= ? AND frequencia >= ?
        ORDER BY nota DESC, id_matricula ASC
        """
        linhas = self.conexao.execute(
            sql, (nota_minima, frequencia_minima)
        ).fetchall()
        return [dict(linha) for linha in linhas]

    def listar_aprovados_por_disciplina(
        self, codigo: str,
        nota_minima=5.0, frequencia_minima=70.0
    ):
        sql = """
        SELECT a.nusp, a.nome, a.curso,
               d.nome_disciplina, m.nota, m.frequencia
        FROM matriculas AS m
        JOIN alunos AS a
            ON m.nusp = a.nusp
        JOIN disciplinas AS d
            ON m.codigo_disciplina = d.codigo_disciplina
        WHERE d.codigo_disciplina = ?
          AND m.nota >= ?
          AND m.frequencia >= ?
        ORDER BY m.nota DESC, a.nome ASC, m.id_matricula ASC
        """
        valores = (codigo, nota_minima, frequencia_minima)
        linhas = self.conexao.execute(sql, valores).fetchall()
        return [dict(linha) for linha in linhas]

    def atualizar(self, matricula: Matricula):
        sql = """
        UPDATE matriculas
        SET periodo = ?, nota = ?, frequencia = ?
        WHERE id_matricula = ?
        """
        valores = (
            matricula.periodo,
            matricula.nota,
            matricula.frequencia,
            matricula.id_matricula
        )
        cursor = self.conexao.execute(sql, valores)
        self.conexao.commit()
        return cursor.rowcount

    def excluir(self, id_matricula: int):
        sql = """
        DELETE FROM matriculas
        WHERE id_matricula = ?
        """
        cursor = self.conexao.execute(sql, (id_matricula,))
        self.conexao.commit()
        return cursor.rowcount


In [ ]:

aluno_dao = AlunoDAO()
disciplina_dao = DisciplinaDAO()
matricula_dao = MatriculaDAO()

# Consultas de alunos
print("Total de alunos:", aluno_dao.contar())
print("Por curso:", aluno_dao.contar_por_curso())
print("Ingressantes em 2025:", aluno_dao.listar_ingressantes(2025))

# Consultas de disciplinas
print("Disciplinas do aluno:", disciplina_dao.buscar_por_nusp(alunos[0].nusp))
print("Alunos de Banco de Dados:",
      disciplina_dao.listar_alunos_matriculados("COMP201"))
print("Médias por disciplina:", disciplina_dao.medias_notas())

# Consultas de matrículas
print("Matrículas do aluno:",
      matricula_dao.listar_por_nusp(alunos[0].nusp))
print("Aprovados:", matricula_dao.listar_aprovados())
print("Aprovados em IA:",
      matricula_dao.listar_aprovados_por_disciplina("COMP301"))


Total de alunos: 60
Por curso: [{'curso': 'Engenharia de Sistemas Digitais', 'quantidade': 17}, {'curso': 'Engenharia Elétrica', 'quantidade': 14}, {'curso': 'Engenharia de Computação', 'quantidade': 13}, {'curso': 'Ciência da Computação', 'quantidade': 10}, {'curso': 'Sistemas de Informação', 'quantidade': 6}]
Ingressantes em 2025: [{'nusp': 95651, 'nome': 'Bernardo Aparecida', 'ano_ingresso': 2025}, {'nusp': 95657, 'nome': 'Brayan Pereira', 'ano_ingresso': 2025}, {'nusp': 95656, 'nome': 'Dr. Gustavo Henrique Rios', 'ano_ingresso': 2025}, {'nusp': 95644, 'nome': 'Eduardo Cunha', 'ano_ingresso': 2025}, {'nusp': 95652, 'nome': 'Heloísa Vargas', 'ano_ingresso': 2025}, {'nusp': 95679, 'nome': 'Juliana Vargas', 'ano_ingresso': 2025}, {'nusp': 95660, 'nome': 'Júlia Porto', 'ano_ingresso': 2025}, {'nusp': 95646, 'nome': 'Luiz Henrique Ferreira', 'ano_ingresso': 2025}, {'nusp': 95670, 'nome': 'Maria Cecília Câmara', 'ano_ingresso': 2025}, {'nusp': 95666, 'nome': 'Natália Casa Grande', 'ano_in

In [ ]:
banco.fechar()
print("Conexão encerrada.")

Conexão encerrada.
